# ASR điền form — Gipformer + Silero VAD
Chọn một trường rồi nói, các đoạn nói nối vào trường đó. Không cần LLM/API key/TTS hoặc đăng nhập.

1. Đẩy code lên GitHub với `Web-ASR` và `Sub-ASR` ở gốc repo.
2. Chọn runtime **T4 GPU**, đặt `REPO_URL`/`REPO_BRANCH` ở cell đầu.
3. Chạy đến cell xem URL, mở link và cấp quyền microphone.
4. Để cell dừng server đến khi sử dụng xong.

Notebook tự clone repo nếu chưa có source; dùng lại checkout có sẵn, không tự pull/ghi đè code. Đây là form liên hệ mẫu: họ tên, số điện thoại, địa chỉ. Dữ liệu ở RAM của phiên; tải JSON để giữ kết quả. Ai có link public đều có thể dùng ứng dụng.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, time, re, signal

FAKE = False  # True: kiểm tra giao diện, transcript giả; không tải model.
SHARE = True
REPO_URL = 'https://github.com/Shinoaki0145/ASR.git'  # Repo chứa Web-ASR và Sub-ASR ở gốc
REPO_BRANCH = ''  # Để trống để dùng nhánh mặc định
REPO_DIR = Path('/content/ASR') if Path('/content').exists() else Path.cwd()
if not (REPO_DIR / 'Web-ASR/pyproject.toml').exists():
    candidates = [Path.cwd(), Path.cwd().parent]
    found = next((p for p in candidates if (p/'Web-ASR/pyproject.toml').exists()), None)
    if found:
        REPO_DIR = found
    elif REPO_URL:
        command = ['git', 'clone'] + (['--branch', REPO_BRANCH] if REPO_BRANCH else []) + [REPO_URL, str(REPO_DIR)]
        subprocess.run(command, check=True)
    else:
        raise RuntimeError('Điền REPO_URL hoặc chạy notebook trong repo đã clone.')
REPO_DIR = REPO_DIR.resolve()
PROJECT = REPO_DIR / 'Web-ASR'
if not (PROJECT/'form_app.py').is_file():
    raise RuntimeError('Checkout chưa có Web-ASR/form_app.py. Đẩy code mới lên GitHub và cập nhật checkout, hoặc chọn REPO_BRANCH đúng rồi clone vào thư mục mới.')
assert (REPO_DIR/'Sub-ASR/voice.py').is_file(), 'Thiếu Sub-ASR; cần repo đầy đủ'
print('Source:', REPO_DIR)


In [ ]:
try:
    from dotenv import dotenv_values
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv>=1,<2'], check=True)
    from dotenv import dotenv_values
candidates = [REPO_DIR/'.env', Path('/content/.env')]
ENV_FILE = Path(os.environ['ASR_ENV_PATH']) if os.environ.get('ASR_ENV_PATH') else next((p for p in candidates if p.is_file()), candidates[0])
SERVER_ENV = {k:v for k,v in dotenv_values(ENV_FILE).items() if v is not None} if ENV_FILE.is_file() else {}
SERVER_ENV.update(os.environ)
SERVER_ENV.update(ASR_ENV_PATH=str(ENV_FILE), WEB_FAKE=str(FAKE).lower(), WEB_SHARE=str(SHARE).lower(), WEB_HOST='0.0.0.0' if SHARE else '127.0.0.1')
SERVER_ENV.setdefault('SEGMENT_SILENCE_SECONDS', '0.7')
print('Đã cấu hình; không cần LLM key hoặc đăng nhập web.')


In [ ]:
if not shutil.which('uv'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
subprocess.run([UV, 'python', 'install', '3.12'], check=True)
EXTRAS = [] if FAKE else ['--extra', 'cuda']
if not FAKE:
    if not shutil.which('nvidia-smi'): raise RuntimeError('Chọn runtime T4 GPU trước')
    subprocess.run(['nvidia-smi'], check=True)
subprocess.run([UV, 'sync', '--locked', *EXTRAS], cwd=PROJECT, check=True)
if not FAKE:
    subprocess.run([UV, 'run', '--locked', *EXTRAS, 'python', '-m', 'sub_asr.setup', '--config', str(REPO_DIR/'Sub-ASR/config.toml')], cwd=PROJECT, env=dict(SERVER_ENV, PYTHONPATH=str(REPO_DIR/'Sub-ASR')), check=True)
print('Môi trường sẵn sàng.')


In [ ]:
def stop_server():
    process = globals().get('FORM_PROCESS')
    if process is not None and process.poll() is None:
        process.send_signal(signal.SIGINT)
        try: process.wait(timeout=12)
        except subprocess.TimeoutExpired:
            process.terminate()
            try: process.wait(timeout=5)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=5)
    log = globals().get('FORM_LOG')
    if log is not None: log.close()
stop_server()
LOG_PATH = PROJECT/'form-server.log'
FORM_LOG = LOG_PATH.open('w')
FORM_PROCESS = subprocess.Popen([UV, 'run', '--locked', *EXTRAS, 'python', '-u', 'form_app.py'], cwd=PROJECT, env=SERVER_ENV, stdout=FORM_LOG, stderr=subprocess.STDOUT)
print('Server PID:', FORM_PROCESS.pid)


In [ ]:
from IPython.display import display, Markdown
for _ in range(25):
    time.sleep(2)
    content = LOG_PATH.read_text(errors='replace')
    urls = re.findall(r'https://[^\s]+\.gradio\.live', content)
    if urls or FORM_PROCESS.poll() is not None: break
print(content[-6000:])
if urls: display(Markdown(f'[Mở ASR điền form]({urls[-1]})'))
print('Server:', 'đang chạy' if FORM_PROCESS.poll() is None else f'đã dừng, exit={FORM_PROCESS.returncode}')


Chọn trường, bật mic và nói. Nghỉ khoảng 0,7 giây để chốt đoạn. Các đoạn tiếp theo nối vào cùng trường. Để sửa/xóa, nhập toàn bộ giá trị ở ô sửa và bấm **Lưu sửa tay**. Tắt mic, chờ xử lý xong và kiểm tra các trường trước khi **Tải JSON**.

Chạy cell dưới khi đã sử dụng xong (dữ liệu chưa tải sẽ mất).


In [ ]:
stop_server()
print('Đã dừng server.')
